<a href="https://colab.research.google.com/github/Bersanone/Unet_skip_conn/blob/main/Unet_con_skip_connections.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Installazione ed import delle dipendenze

!pip install albumentations
!pip install torchmetrics


import os
from torch import nn
from torchvision import models
from torch.nn.functional import relu
import torchmetrics
import numpy as np
from PIL import Image
#Importiamo albumentations serve per gestire la sincronizzazione tra imaggine e maschere nella data argumentation
import albumentations as A
import cv2

import torchvision.transforms.functional as TF


#Impostiamo le variabili d'ambiente per kaggle
os.environ['KAGGLE_USERNAME'] = ""
os.environ['KAGGLE_KEY'] = ""

In [ ]:
!kaggle datasets download ipythonx/celebamaskhq

In [ ]:
!unzip /content/celebamaskhq.zip

Script per unificare le maschere

In [ ]:
#Creiamo un array contente i nomi delle maschere specifiche

label_list = ['skin', 'neck', 'cloth', 'hair', 'l_ear', 'r_ear',
    'l_brow', 'r_brow', 'nose', 'mouth', 'u_lip', 'l_lip',
    'l_eye', 'r_eye', 'eye_g', 'ear_r', 'neck_l', 'hat']


#Path delle directory
folder_base = "/content/CelebAMask-HQ/CelebAMask-HQ-mask-anno"
folder_save = "CelebAMaskHQ-mask"

#Numero totale delle immagini
img_num = 30000


#Utilizziamo quetso comando per entare nella directory corrente
os.chdir(os.path.join(os.getcwd()))

#Ls della directory
print(os.listdir())


#Funzione per creare una nuova cartella
def make_folder(path):
  if not os.path.exists(os.path.join(path)):
    os.makedirs(os.path.join(path))


make_folder(folder_save)

print("inizio ciclo")


#Per ogni immaginr
for i in range(img_num):
  #Dividiamo per 2000 perchè ogni subdirectory di mask contiene le mask di 2000 individui
  folder_nr = i//2000
  #Eseguaiamo un resize su un tensore di 0
  im_base = np.zeros((512,512), dtype=np.uint8)

  is_empty = True



  for idx,label in enumerate(label_list):
    #Costruiamo il path delle maschere singole
    file_name = os.path.join(folder_base, str(folder_nr),str(i).rjust(5, '0') + '_'+ label + '.png')
    #Verifichiamo l'esistenza
    if os.path.exists(file_name.strip(' ')):
      #Se esiste impostiamo is_empty a false
      is_empty = False
      #Carichiamo l'immagine in scala di grigi
      im = cv2.imread(file_name, cv2.IMREAD_GRAYSCALE)
      #Assegniamo l'indice alla classe
      C = idx + 1
      #Se l'immagine è diversa da 0 assegnamo la relativa classe
      im_base[im != 0] = C
  #Se non è vuota creaiamo una nuova maschera
  if not is_empty:
    filename_save = os.path.join(folder_save, str(i) + '.png')
    print(filename_save)
    #Salviamo la maschera sui tensori di 0
    cv2.imwrite(filename_save, im_base)


Streaming output truncated to the last 5000 lines.
CelebAMaskHQ-mask/25000.png
CelebAMaskHQ-mask/25001.png
CelebAMaskHQ-mask/25002.png
CelebAMaskHQ-mask/25003.png
CelebAMaskHQ-mask/25004.png
CelebAMaskHQ-mask/25005.png
CelebAMaskHQ-mask/25006.png
CelebAMaskHQ-mask/25007.png
CelebAMaskHQ-mask/25008.png
CelebAMaskHQ-mask/25009.png
CelebAMaskHQ-mask/25010.png
CelebAMaskHQ-mask/25011.png
CelebAMaskHQ-mask/25012.png
CelebAMaskHQ-mask/25013.png
CelebAMaskHQ-mask/25014.png
CelebAMaskHQ-mask/25015.png
CelebAMaskHQ-mask/25016.png
CelebAMaskHQ-mask/25017.png
CelebAMaskHQ-mask/25018.png
CelebAMaskHQ-mask/25019.png
CelebAMaskHQ-mask/25020.png
CelebAMaskHQ-mask/25021.png
CelebAMaskHQ-mask/25022.png
CelebAMaskHQ-mask/25023.png
CelebAMaskHQ-mask/25024.png
CelebAMaskHQ-mask/25025.png
CelebAMaskHQ-mask/25026.png
CelebAMaskHQ-mask/25027.png
CelebAMaskHQ-mask/25028.png
CelebAMaskHQ-mask/25029.png
CelebAMaskHQ-mask/25030.png
CelebAMaskHQ-mask/25031.png
CelebAMaskHQ-mask/25032.png
CelebAMaskHQ-mask/25033.p

Preparazione del ds

In [ ]:
import torch

#Creiamo il dataset istanziando la relativa classe in torch

class faceFeatures(torch.utils.data.Dataset):
  def __init__(self):
    #Recuperiamo i path delle immagini e delle maschere
    self.image_paths = "/content/CelebAMask-HQ/CelebA-HQ-img"
    self.mask_paths = "/content/CelebAMaskHQ-mask"

    self.image_files = sorted([f for f in os.listdir(self.image_paths) if f.endswith(".jpg")],
                              #Restituiamo solo il numero del file:
                              #(0.jpg restituira 0)
                              key = lambda x: int(x.split(".")[0])

                              )


    self.parts = [
            'skin',      # 1
            'l_brow',    # 2
            'r_brow',    # 3
            'l_eye',     # 4
            'r_eye',     # 5
            'eye_g',     # 6 (Occhiali)
            'l_ear',     # 7
            'r_ear',     # 8
            'ear_r',     # 9 (Orecchini)
            'nose',      # 10
            'mouth',     # 11
            'u_lip',     # 12
            'l_lip',     # 13
            'neck',      # 14
            'neck_l',    # 15 (Collana)
            'cloth',     # 16
            'hair',      # 17
            'hat'        # 18
        ]


  def __len__(self):

    #Restituiamo il numero di immagini per path

    return len(self.image_files)



  def __getitem__(self,idx):

    #Selezioniamol'immagine con l'indice
    img_name = self.image_files[idx]
    #Indice della classe
    base_idx = int(img_name.split(".")[0])



    #carichiamo l'immagine in RGB


    image_path = os.path.join(self.image_paths, img_name)
    image = Image.open(image_path).convert("RGB")


    #Selezioniamo la maschera

    mask_prefix = f"{base_idx}.png"

    #Selezioniamola dalla folder

    mask_folder = os.path.join(self.mask_paths, mask_prefix)

# Carichiamo come immagine in scala di grigi ("L")
    mask = Image.open(mask_folder).convert("L")


    #Resize delle immagini

    #Utilizziamo BILINEAR(interpolazione Bilaterale) nelle immagini, i pixel che non esistono dopo il rescale vengono calcolati con la media dei 4 pixel più vicini

    image = image.resize((256,256), Image.BILINEAR)

    #Utilizziamo la interpolazione nearest nelle maschere cossihè i nuovi pixel prendano lo stesso valore del pixel prima

    mask = mask.resize((256,256), Image.NEAREST)

    #Addattiamo i canali dell'immagine per pytorch
    image_tensor = TF.to_tensor(image)
    #Carichiamo le maschere come array numpy e li convertiamo in un intero da 64
    mask_np = np.array(mask, dtype=np.int64)

    #Passiamo le maschere da numpy a torch

    mask_tensor = torch.from_numpy(mask_np)


    #Restituiamo le imamgini

    return image_tensor, mask_tensor













In [ ]:
#Istanziamo il dataset

ds = faceFeatures()

Struttura del modello con skip connections

In [ ]:
class UnetModel(nn.Module):

  def __init__(self,num_class):

    #Istanziamo il costruttore della classe genitore

    super().__init__()



    #Sezione di Encoder

    #Sezione encoder 1

    #Primo layer Conv2d che prende in ingtresso 3 canali e ne restituisce 64, utilizziamo un kernel 3*3 con il padding=1

    self.e11 = nn.Conv2d(3,64,kernel_size=3,padding=1)

    self.e12 = nn.Conv2d(64,64,kernel_size=3,padding=1)

    #Utilizziamo une kernel_size = 2 e uno stride di 2 per dividere a metà le immagini (la finestra è larga 2 pixel e scorre di due pixel)
    self.pool1 = nn.MaxPool2d(kernel_size=2,stride=2)


    #Sezione encoder 2

    self.e21 = nn.Conv2d(64,128,kernel_size=3,padding=1)
    self.e22 = nn.Conv2d(128,128,kernel_size=3,padding=1)
    self.pool2 = nn.MaxPool2d(kernel_size=2,stride=2)


    #Sezione encoder 3

    self.e31 = nn.Conv2d(128,256,kernel_size=3,padding=1)
    self.e32 = nn.Conv2d(256,256,kernel_size=3,padding=1)
    self.pool3 = nn.MaxPool2d(kernel_size=2,stride=2)



    #Sezione encoder 4

    self.e41 = nn.Conv2d(256,512,kernel_size=3,padding=1)
    self.e42 = nn.Conv2d(512,512,kernel_size=3,padding=1)
    self.pool4 = nn.MaxPool2d(kernel_size=2,stride=2)



    #Sezione encoder 5


    self.e51 = nn.Conv2d(512,1024,kernel_size=3,padding=1)
    self.e52 = nn.Conv2d(1024,1024,kernel_size=3,padding=1)




    #Sezione decoder 1

    #Utilizziamo kernel_size e stride a 2 per raddoppiare la dimensione dell'immagine e dimezzarne i canali
    #Utilizziamo la fase di transponse per ringrandire l'immagine
    #Ogni transponse ha i relativi encoder, in questo modo teniamo traccia della posizione degli oggetti

    self.upconv1 = nn.ConvTranspose2d(1024,512,kernel_size=2,stride=2)
    self.d11 = nn.Conv2d(1024,512,kernel_size=3,padding=1)
    self.d12 = nn.Conv2d(512,512,kernel_size=3,padding=1)

    #Sezione decoder 2


    self.upconv2 = nn.ConvTranspose2d(512,256,kernel_size=2,stride=2)
    self.d21 = nn.Conv2d(512,256,kernel_size=3,padding=1)
    self.d22 = nn.Conv2d(256,256,kernel_size=3,padding=1)


    #Sezione decoder 3


    self.upconv3 = nn.ConvTranspose2d(256,128,kernel_size=2,stride=2)
    self.d31 = nn.Conv2d(256,128,kernel_size=3,padding=1)
    self.d32 = nn.Conv2d(128,128,kernel_size=3,padding=1)



    #Sezione decoder 4


    self.upconv4 = nn.ConvTranspose2d(128,64,kernel_size=2,stride=2)
    self.d41 = nn.Conv2d(128,64,kernel_size=3,padding=1)
    self.d42 = nn.Conv2d(64,64,kernel_size=3,padding=1)


    #Output layer


    self.output = nn.Conv2d(64,num_class,kernel_size=1)


    #Forward

  #Passo di forward

  def forward(self,x):

      #Encoder

      #Aggiungiamo le attivazioni RELU agli encoder
      #Applichiamo il max pooling alla seconda attivazione

      xe11 = relu(self.e11(x))
      xe12 = relu(self.e12(xe11))
      xp1 = self.pool1(xe12)



      xe21 = relu(self.e21(xp1))
      xe22 = relu(self.e22(xe21))
      xp2 = self.pool2(xe22)



      xe31 = relu(self.e31(xp2))
      xe32 = relu(self.e32(xe31))
      xp3 = self.pool3(xe32)



      xe41 = relu(self.e41(xp3))
      xe42 = relu(self.e42(xe41))
      xp4= self.pool4(xe42)

      #Non applichiamo il pooling per preservare le informazioni di posizione



      xe51 = relu(self.e51(xp4))
      xe52 = relu(self.e52(xe51))




      #Decoder


      xu1 = self.upconv1(xe52)
      #Utilizziamo torch.cat per unire le informazioni del decoder con quella dell'encoderù

      #Esempio:


      #   x = torch.tensor([[1, 2, 3],
      #               [4, 5, 6]])

      #  y = torch.tensor([[7, 8, 9],
      #               [10, 11, 12]])

      #torch.cat((x, y), dim=0)

      # Risultato:

        # [[ 1,  2,  3],
        #  [ 4,  5,  6],
        #  [ 7,  8,  9],
        #  [10, 11, 12]]

      xu11 = torch.cat([xu1,xe42],dim=1)
      xd11 = relu(self.d11(xu11))
      xd12 = relu(self.d12(xd11))



      xu2 = self.upconv2(xd12)
      xu22 = torch.cat([xu2,xe32], dim=1)
      xd21 = relu(self.d21(xu22))
      xd22 = relu(self.d22(xd21))


      xu3 = self.upconv3(xd22)
      xu32 = torch.cat([xu3,xe22], dim=1)
      xd31 = relu(self.d31(xu32))
      xd32 = relu(self.d32(xd31))



      xu4 = self.upconv4(xd32)
      xu42 = torch.cat([xu4,xe12], dim=1)
      xd41 = relu(self.d41(xu42))
      xd42 = relu(self.d42(xd41))


      #Layer di output


      out = self.output(xd42)


      return out






In [ ]:
#Istanziamo il modello con 19 classi

model = UnetModel(num_class=19)

Training del modello

In [ ]:
#Istanziamo la lunhezza del dataset

ds_len = len(ds)

#Selezioniamo il 75% dei dati come training

train_len = int(0.75*ds_len)

#Il resto dei dati viene salvato come test

test_len = ds_len - train_len

#Dividiamo i dataset

train_ds,test_ds = torch.utils.data.random_split(ds,[train_len,test_len])



train_dataloader = torch.utils.data.DataLoader(train_ds,batch_size=32,shuffle=True)
val_dataloader = torch.utils.data.DataLoader(test_ds,batch_size=32,shuffle=False)

In [ ]:
from tqdm import tqdm

#Creiamo una funzione per calcolare i pesi del dataset

def pesi_classi(dataloader, num_classi=19):
  #Creiamo un tensore di 0 per numero di classi
  pixel_per_classe = torch.zeros(num_classi)
  #Iteriamo sul dataloader
  for _, ybatch in tqdm(dataloader, desc="Scansione Dataset"):
    #Appiattiamo tutte le maschere ad una dimensione
    flatted = ybatch.view(-1)

    #Torch.bincount serve per contare quante volte appare un classe all'interno del tensore flatten
    #Specificando minlength=num_classi, forzi PyTorch a restituire un tensore lungo almeno quanto il numero delle tue classi,
    #riempiendo di zero le classi che non sono apparse.
    count = torch.bincount(flatted, minlength=num_classi)
    pixel_per_classe += count

  #Settiamo a 1 i pixel a 0 in modo da eviatre errori in caso di divisione

  pixel_per_classe[pixel_per_classe==0] = 1.0

  #Calcoliamo la prob
  prob = pixel_per_classe / pixel_per_classe.sum()

  #Utilizziamo questo modello per distribuire equamente l'importanza
  #Una classe molto frequente deve avere meno peso nella funzione di loss in modo da non sbilanciare l'addestramento
  #Utilizziamo il logaritmo per far schiacciare io pesi grandi essendo che se dividessimo 1.0/(prob=0.1) otterremo 1000 facebndo saltare il gradiente

  #Sommiamo 1.02 invece di 1.0 oppure di tenere solo log perchè causerebbe un valore tendente all'infinto facendo saltare il gradiente

  #Con 1.02 garantiamo un limite che da possibilità al modello di dare importanza alle classi più rare ma non di esagerare


  pesi = 1.0/ torch.log(1.02 + prob)

  return pesi




In [ ]:
from torchmetrics import JaccardIndex
print("inizio")
LR = 1e-3
EPOCHS = 300

#Passiamo i parametri da ottimizzare al modello e impostiamo manualmente il learning rate
optimizer = torch.optim.Adam(model.parameters(),lr=LR, weight_decay=1e-4)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)


patience = 0

best_iou = 0.0

#Carichiamo il modello sulla gpu

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


#Impostiamo la IOU con il nostro numero di classi
jaccard_train = JaccardIndex(task="multiclass", num_classes=19).to(device)
jaccard_val = JaccardIndex(task="multiclass", num_classes=19).to(device)

pesi = pesi_classi(train_dataloader).to(device)

#Usiamo la crossentropy essendo classi multiple
loss = nn.CrossEntropyLoss(weight=pesi)

print("2. Inizio ciclo epoche...")

for epoch in range(EPOCHS):
  model.train()
  #Variabile per immagazzinare i valori delle loss
  e_loss = 0.0
  e_acc = 0.0
  loop_train = tqdm(train_dataloader, leave=True, desc=f"Training Epoch {epoch+1}/{EPOCHS}")
  for xbatch,ybatch in loop_train:
    xbatch = xbatch.to(device)
    ybatch = ybatch.to(device)
    optimizer.zero_grad()
    pred = model(xbatch)

    loss_v = loss(pred,ybatch)
    loss_v.backward()
    optimizer.step()
    e_loss += loss_v.item()

    accuracy = (pred.argmax(dim=1) == ybatch).float().mean()


    jaccard_train(pred, ybatch)

    e_acc += accuracy.item()
  #Calcoliamo la media
  avg_loss = e_loss/len(train_dataloader)
  avg_acc = e_acc/len(train_dataloader)
  iou_epoch_train = jaccard_train.compute()
  jaccard_train.reset()

  model.eval()

  v_loss = 0.0
  v_acc = 0.0


  with torch.no_grad():
    for xbatch,ybatch in val_dataloader:
      xbatch = xbatch.to(device)
      ybatch = ybatch.to(device)

      pred = model(xbatch)

      loss_v = loss(pred,ybatch)

      v_loss += loss_v.item()

      acc_v = (pred.argmax(dim=1) == ybatch).float().mean()

      v_acc += acc_v.item()

      jaccard_val(pred,ybatch)

  avg_v_loss = v_loss/len(val_dataloader)
  avg_v_acc = v_acc/len(val_dataloader)

  iou_epoch_val = jaccard_val.compute()
  jaccard_val.reset()

  scheduler.step(iou_epoch_val)

  #Stampiamo per ogni epoca
  print(f"Epochs: {epoch+1}/{EPOCHS} | "
        f"Train_loss: {avg_loss:.4f}, Train_accuracy: {avg_acc*100:.2f}, IOU_train: {iou_epoch_train*100:.2f}% | "
        f" Val_loss: {avg_v_loss:.4f}, Val_accuracy: {avg_v_acc*100:.2f}, IOU_val: {iou_epoch_val*100:.2f}% | ")


  if iou_epoch_val > best_iou:
    best_iou = iou_epoch_val
    patience = 0

    torch.save(model.state_dict(), "best_version_segmentation.pth")


  else:

    patience +=1

  if patience == 20:
    break



inizio


Scansione Dataset: 100%|██████████| 704/704 [05:22<00:00,  2.18it/s]


2. Inizio ciclo epoche...


Training Epoch 1/300:  31%|███       | 218/704 [07:23<16:23,  2.02s/it]

In [ ]:
from google.colab import files

files.download("best_version_segmentation.pth")